**DSAI 305 — Phase 2 | Team 505**  
**Task:** 12-Class Lung Disease Classification on NIH ChestX-ray14

| Item | Detail |
|------|--------|
| Architecture | DenseNet-121 (Huang et al., CVPR 2017) |
| Pretrained weights | ImageNet-1K (torchvision) |
| CXR reference | Rajpurkar et al., CheXNet, 2017 |

---
## 1 — Imports

In [11]:
import time
import cv2
import os, time, cv2, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models



import warnings
warnings.filterwarnings('ignore')

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device('cuda' if CUDA_AVAILABLE else 'cpu')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = CUDA_AVAILABLE

print(f'PyTorch: {torch.__version__} | CUDA: {CUDA_AVAILABLE} | Device: {DEVICE}')
if CUDA_AVAILABLE:
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from sklearn.metrics import auc as sk_auc
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, f1_score,
    precision_score, recall_score, confusion_matrix,
    accuracy_score, classification_report
)

PyTorch: 2.12.0+cu126 | CUDA: True | Device: cuda
GPU: NVIDIA GeForce RTX 3070 Laptop GPU


---
## 2 — Config

In [12]:
RUN_MODE = "full"

IMG_SIZE   = 288
BATCH_SIZE = 16

_EPOCH_MAP = {"smoke": 1, "dev": 15, "full": 50}
NUM_EPOCHS = _EPOCH_MAP[RUN_MODE]

EARLY_STOP_PATIENCE = 20
LABEL_SMOOTHING     = 0.05   # prevents overconfidence
MIXUP_ALPHA         = 0.3    # mixup strength (0=off, higher=more mixing)
EARLY_STOP_ENABLED  = True

WARMUP_EPOCHS          = 2
PARTIAL_UNFREEZE_EPOCH = 3
FULL_UNFREEZE_EPOCH    = 4

HEAD_LR      = 1e-4
PARTIAL_LR   = 2e-5
FULL_LR      = 2e-5
WEIGHT_DECAY = 3e-3

NUM_WORKERS = 0
PIN_MEMORY  = CUDA_AVAILABLE

PROJECT_ROOT = Path('..').resolve()
DATA_SPLITS  = PROJECT_ROOT / 'data' / 'splits'
OUTPUT_DIR   = PROJECT_ROOT / 'outputs' / 'DenseNet121'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Mode: {RUN_MODE.upper()} | Epochs: {NUM_EPOCHS} | Patience: {EARLY_STOP_PATIENCE}')
print(f'IMG: {IMG_SIZE} | BS: {BATCH_SIZE} | WD: {WEIGHT_DECAY}')
print(f'Unfreeze: A(1-2) B({PARTIAL_UNFREEZE_EPOCH}) C({FULL_UNFREEZE_EPOCH})')

Mode: FULL | Epochs: 50 | Patience: 20
IMG: 288 | BS: 16 | WD: 0.003
Unfreeze: A(1-2) B(3) C(4)


---
## 3 — Data Loading

---
## 3a — Balanced Dataset Construction

In [13]:
if RUN_MODE == "full":
    df_train = pd.read_csv(DATA_SPLITS / 'train.csv')
    df_val   = pd.read_csv(DATA_SPLITS / 'val.csv')
    df_test  = pd.read_csv(DATA_SPLITS / 'test.csv')
elif RUN_MODE == "dev":
    df_train = pd.read_csv(DATA_SPLITS / 'train.csv')
    df_val   = pd.read_csv(DATA_SPLITS / 'val.csv')
    df_test  = pd.read_csv(DATA_SPLITS / 'test.csv')
    df_train = df_train.sample(frac=0.2, random_state=42).reset_index(drop=True)
else:
    raise ValueError(f'Unknown RUN_MODE: {RUN_MODE}')

# Normalize label column
for _df in [df_train, df_val, df_test]:
    if 'label' in _df.columns:
        _df['label'] = _df['label'].astype(int)
    if 'source_weight' not in _df.columns:
        _df['source_weight'] = 1.0

_pos = 'label'
print(f'Train: {len(df_train):,} samples across {df_train[_pos].nunique()} classes')
print(f'Val : {len(df_val):,} samples')
print(f'Test : {len(df_test):,} samples')
print(f'Val  : {len(df_val):,}   | Pos: {df_val[_pos].sum():.0f}   | Rate: {df_val[_pos].mean()*100:.1f}%')
print(f'Test : {len(df_test):,}  | Pos: {df_test[_pos].sum():.0f}  | Rate: {df_test[_pos].mean()*100:.1f}%')


Train: 11,943 samples across 12 classes
Val : 2,611 samples
Test : 2,618 samples
Val  : 2,611   | Pos: 14511   | Rate: 555.8%
Test : 2,618  | Pos: 14688  | Rate: 561.0%


---
## 4 — Transforms

In [14]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
TRAIN_RESIZE  = 320

class CLAHETransform:
    """CLAHE with pre-downscale guard — prevents MemoryError on full-res NIH X-rays."""
    def __call__(self, pil_img):
        if max(pil_img.size) > 512:
            pil_img = pil_img.resize((512, 512), Image.LANCZOS)
        img_np   = np.array(pil_img.convert('L'))
        clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(img_np)
        return Image.fromarray(enhanced).convert('RGB')

train_transforms = transforms.Compose([
    CLAHETransform(),
    transforms.Resize(TRAIN_RESIZE),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05),
                            scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.20, contrast=0.20),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    CLAHETransform(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Train: CLAHE->Resize(320)->RandomCrop(288)->Flip->Rotate->Jitter->Norm')
print('Val  : CLAHE->Resize(288)->Norm')

Train: CLAHE->Resize(320)->RandomCrop(288)->Flip->Rotate->Jitter->Norm
Val  : CLAHE->Resize(288)->Norm


---
## 5 — Dataset

In [15]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.image_paths = self.df['image_path'].tolist()
        self.labels = self.df['label'].values.astype(np.int64)
        self.weights = (
            self.df['source_weight'].values.astype(np.float32)
            if 'source_weight' in self.df.columns
            else np.ones(len(self.df), dtype=np.float32)
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label  = self.labels[idx]
        weight = self.weights[idx]
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))
        if self.transform:
            try:
                img = self.transform(img)
            except MemoryError:
                img = torch.zeros(3, IMG_SIZE, IMG_SIZE)
        return img, label, weight

---
## 6 — Model

In [16]:
model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)

num_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(num_features, 12)
)

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: DenseNet121 | Params: {total_params:,} | Trainable: {trainable:,} | Device: {DEVICE}')

Model: DenseNet121 | Params: 6,966,156 | Trainable: 6,966,156 | Device: cuda


---
## 7 — Loss, Sampler & Optimizer

In [17]:
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, num_classes=12, smoothing=0.05):
        super().__init__()
        self.smoothing = smoothing
        self.num_classes = num_classes

    def forward(self, inputs, targets):
        n = self.num_classes
        log_probs = F.log_softmax(inputs, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=n).float()
        targets_smooth = (targets_one_hot * (1 - self.smoothing)
                          + (1 - targets_one_hot) * self.smoothing / (n - 1))
        loss = -(targets_smooth * log_probs).sum(dim=1)
        return loss

criterion = LabelSmoothingCrossEntropy(num_classes=12, smoothing=LABEL_SMOOTHING)

labels_array = df_train['label'].values
class_counts = np.bincount(labels_array)
sample_weights = 1.0 / class_counts[labels_array]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float64),
    num_samples=len(sample_weights), replacement=True
)

train_dataset = ChestXrayDataset(df_train, transform=train_transforms)
val_dataset = ChestXrayDataset(df_val, transform=val_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

print(f'Train loader: {len(train_dataset):,} samples | {len(train_loader):,} batches')
print(f'Val loader: {len(val_dataset):,} samples')
print(f'Loss: LabelSmoothingCrossEntropy(12, {LABEL_SMOOTHING}) | Sampler: balanced')
print(f'Train: {len(train_dataset):,} -> {len(train_loader):,} batches')
print(f'Val: {len(val_dataset):,} -> {len(val_loader):,} batches')
print(f'Stage A: head-only warmup for {WARMUP_EPOCHS} epochs')


Train loader: 11,943 samples | 747 batches
Val loader: 2,611 samples
Loss: LabelSmoothingCrossEntropy(12, 0.05) | Sampler: balanced
Train: 11,943 -> 747 batches
Val: 2,611 -> 164 batches
Stage A: head-only warmup for 2 epochs


---
## 8 — Train & Validate Functions

In [18]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    n_batches = 0
    for images, labels, weights in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).long()
        weights = weights.to(device, non_blocking=True).unsqueeze(1)
        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = (criterion(outputs, labels).unsqueeze(1) * weights).mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
        n_batches += 1
    return running_loss / max(n_batches, 1)

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    n_batches = 0
    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels, _ in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).long()
            outputs = model(images)
            loss = criterion(outputs, labels).mean()
            running_loss += loss.item()
            n_batches += 1
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy().tolist())
    all_labels = np.array(all_labels, dtype=np.int64)
    all_probs = np.array(all_probs, dtype=np.float32)
    all_preds = all_probs.argmax(axis=1)
    return running_loss / max(n_batches, 1), all_preds, all_probs, all_labels

print('Train/validate functions defined.')


Train/validate functions defined.


---
## 9 — Training Loop

In [19]:
PARTIAL_LAYERS = ['denseblock4', 'norm5', 'classifier']

In [20]:
def tune_threshold(probs, labels):
    """Multi-class: argmax prediction, macro F1."""
    preds = probs.argmax(axis=1)
    return f1_score(labels, preds, average='macro'), 0.0

history = {
    'train_loss': [], 'val_loss': [],
    'val_auc': [], 'val_f1': [],
    'stage': [], 'lr': []
}
best_score        = (-1.0, -1.0)
best_epoch        = 0
epochs_no_improve = 0
best_model_path   = OUTPUT_DIR / 'best_model.pth'
best_checkpoint   = None

print('=' * 70)
print(f'TRAINING DENSENET121 | {RUN_MODE.upper()} MODE')
print('=' * 70)

start_time = time.time()

# Stage A: freeze backbone, train head only
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True
optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=HEAD_LR, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=WARMUP_EPOCHS, eta_min=1e-6
)
_current_stage = 'A'

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()
    if epoch == PARTIAL_UNFREEZE_EPOCH:
        for param in model.parameters():
            param.requires_grad = False
        for name, param in model.named_parameters():
            if any(x in name for x in PARTIAL_LAYERS):
                param.requires_grad = True
        trainable_p = [p for p in model.parameters() if p.requires_grad]
        optimizer   = optim.AdamW(trainable_p, lr=PARTIAL_LR, weight_decay=WEIGHT_DECAY)
        remaining   = NUM_EPOCHS - PARTIAL_UNFREEZE_EPOCH + 1
        scheduler   = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=remaining, eta_min=1e-6)
        _current_stage = 'B'

    elif epoch == FULL_UNFREEZE_EPOCH:
        for param in model.parameters():
            param.requires_grad = True
        optimizer  = optim.AdamW(model.parameters(), lr=FULL_LR, weight_decay=WEIGHT_DECAY)
        remaining  = NUM_EPOCHS - FULL_UNFREEZE_EPOCH + 1
        # CyclicLR: resets every 6 epochs, prevents LR collapse + memorization
        scheduler  = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=6,        # restart every 6 epochs
            T_mult=1,     # keep cycle length constant
            eta_min=1e-6
        )
        _current_stage = 'C'

    if _current_stage == 'A': _stage_label = 'STAGE A: Head Warmup'
    elif _current_stage == 'B': _stage_label = 'STAGE B: Partial Fine-tune'
    elif _current_stage == 'C': _stage_label = 'STAGE C: Full Fine-tune'

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_preds_05, val_probs, val_labels = validate(model, val_loader, criterion, DEVICE)

            # Multi-class AUC: one-vs-rest macro
    auc_roc = roc_auc_score(val_labels, val_probs, multi_class='ovr', average='macro')
    auc_pr = average_precision_score(val_labels, val_probs, average='macro')
    

    acc_05  = accuracy_score(val_labels, val_preds_05)
    prec_05 = precision_score(val_labels, val_preds_05, average="macro", zero_division=0)
    rec_05  = recall_score(val_labels, val_preds_05, average='macro', zero_division=0)
    f1_05   = f1_score(val_labels, val_preds_05, average='macro', zero_division=0)

    best_f1_ep, best_thr_ep = tune_threshold(val_probs, val_labels)
    val_preds_tuned = val_probs.argmax(axis=1)
    prec_tuned = precision_score(val_labels, val_preds_tuned, average='macro', zero_division=0)
    rec_tuned  = recall_score(val_labels, val_preds_tuned, average='macro', zero_division=0)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(auc_roc)
    history['val_f1'].append(best_f1_ep)

    dur = time.time() - epoch_start
    lr_curr = optimizer.param_groups[0]['lr']

    print(f'\nEpoch {epoch:2d}/{NUM_EPOCHS} | {_stage_label} | LR: {lr_curr:.2e} | {dur:.0f}s')
    print(f'  LOSS: Train={train_loss:.4f} | Val={val_loss:.4f}')
    print(f'  AUC : ROC={auc_roc:.4f} | PR={auc_pr:.4f}')
    print(f'  @0.5: Acc={acc_05:.4f} | P={prec_05:.4f} | R={rec_05:.4f} | F1={f1_05:.4f} | Pred+={int(val_preds_05.sum())}')
    print(f'  BEST: F1={best_f1_ep:.4f} @ Thr={best_thr_ep:.3f} | P={prec_tuned:.4f} | R={rec_tuned:.4f} | Pred+={int(val_preds_tuned.sum())}')

    current_score = (auc_roc, auc_pr)
    if current_score > best_score:
        best_score = current_score
        best_epoch = epoch
        epochs_no_improve = 0
        best_checkpoint = {'auc_roc': auc_roc, 'f1': best_f1_ep,
            'accuracy': acc_05,
                           }
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                     'val_auc': auc_roc}, best_model_path)
        print(f'  [NEW BEST] Epoch {epoch} | ROC-AUC={auc_roc:.4f} | PR-AUC={auc_pr:.4f}')
    else:
        epochs_no_improve += 1

    if scheduler is not None:
        scheduler.step()
    history['stage'].append(_current_stage)
    history['lr'].append(lr_curr)

    if EARLY_STOP_ENABLED and epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

total_time = (time.time() - start_time) / 60
print(f'\nTraining complete. {total_time:.1f} min. Best epoch: {best_epoch}')

TRAINING DENSENET121 | FULL MODE

Epoch  1/50 | STAGE A: Head Warmup | LR: 1.00e-04 | 515s
  LOSS: Train=2.5298 | Val=2.4583
  AUC : ROC=0.5970 | PR=0.1205
  @0.5: Acc=0.1164 | P=0.1804 | R=0.1207 | F1=0.0773 | Pred+=9286
  BEST: F1=0.0773 @ Thr=0.000 | P=0.1804 | R=0.1207 | Pred+=9286
  [NEW BEST] Epoch 1 | ROC-AUC=0.5970 | PR-AUC=0.1205

Epoch  2/50 | STAGE A: Head Warmup | LR: 5.05e-05 | 486s
  LOSS: Train=2.4915 | Val=2.4256
  AUC : ROC=0.6203 | PR=0.1318
  @0.5: Acc=0.1444 | P=0.1525 | R=0.1446 | F1=0.1230 | Pred+=12941
  BEST: F1=0.1230 @ Thr=0.000 | P=0.1525 | R=0.1446 | Pred+=12941
  [NEW BEST] Epoch 2 | ROC-AUC=0.6203 | PR-AUC=0.1318

Epoch  3/50 | STAGE B: Partial Fine-tune | LR: 2.00e-05 | 477s
  LOSS: Train=2.4482 | Val=2.3795
  AUC : ROC=0.6700 | PR=0.1622
  @0.5: Acc=0.1620 | P=0.1761 | R=0.1633 | F1=0.1243 | Pred+=12345
  BEST: F1=0.1243 @ Thr=0.000 | P=0.1761 | R=0.1633 | Pred+=12345
  [NEW BEST] Epoch 3 | ROC-AUC=0.6700 | PR-AUC=0.1622

Epoch  4/50 | STAGE C: Full Fine

---
## 10 — Training Curves

In [21]:
n_ep = len(history['train_loss'])
if n_ep >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    ep_range = range(1, n_ep + 1)
    axes[0].plot(ep_range, history['train_loss'], 'o-', label='Train', color='#3498db')
    axes[0].plot(ep_range, history['val_loss'], 'o-', label='Val', color='#e74c3c')
    if best_epoch > 0: axes[0].axvline(best_epoch, color='green', ls='--', alpha=0.5, label=f'Best (ep {best_epoch})')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss Curves', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(ep_range, history['val_auc'], 's-', color='#2ecc71')
    if best_epoch > 0: axes[1].axvline(best_epoch, color='green', ls='--', alpha=0.5)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('ROC-AUC')
    axes[1].set_title('Validation AUC', fontweight='bold'); axes[1].grid(alpha=0.3)
    axes[2].plot(ep_range, history['val_f1'], 'd-', color='#9b59b6')
    if best_epoch > 0: axes[2].axvline(best_epoch, color='green', ls='--', alpha=0.5)
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('F1-Score')
    axes[2].set_title('Validation F1', fontweight='bold'); axes[2].grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f'[SAVED] {OUTPUT_DIR / "training_curves.png"}')

[SAVED] C:\RIE\outputs\DenseNet121\training_curves.png


---
## 11 — Evaluation

In [22]:
checkpoint = torch.load(best_model_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
val_loss, val_preds_d, val_probs, val_labels = validate(model, val_loader, criterion, DEVICE)

val_preds = val_probs.argmax(axis=1)
acc  = accuracy_score(val_labels, val_preds)
prec = precision_score(val_labels, val_preds, average="macro", zero_division=0)
rec  = recall_score(val_labels, val_preds, average="macro", zero_division=0)
f1   = f1_score(val_labels, val_preds, average="macro", zero_division=0)
auc  = roc_auc_score(val_labels, val_probs, multi_class='ovr', average='macro')
single_roc = auc
single_pr = average_precision_score(val_labels, val_probs, average='macro')
single_f1 = f1

print('=' * 60)
print(f'DENSENET121 VALIDATION (epoch {best_epoch})')
print('=' * 60)
print(f'  ROC-AUC   : {auc:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  Accuracy  : {acc:.4f}')
print('=' * 60)
CLASS_NAMES = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
                'Effusion', 'Emphysema', 'Infiltration', 'Mass',
                'No Finding', 'Nodule', 'Pneumonia', 'Pneumothorax']


DENSENET121 VALIDATION (epoch 21)
  ROC-AUC   : 0.7888
  F1-Score  : 0.3394
  Precision : 0.3479
  Recall    : 0.3391
  Accuracy  : 0.3405


In [23]:
from sklearn.metrics import confusion_matrix
CLASS_NAMES = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Infiltration', 'Mass', 'No Finding', 'Nodule', 'Pneumonia', 'Pneumothorax']
cm = confusion_matrix(val_labels, val_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix (DenseNet121, thr=0.0)\\nAcc={acc:.3f}  F1={f1:.3f}  AUC={auc:.3f}',
    fontweight='bold')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print(f'[SAVED] {OUTPUT_DIR / "confusion_matrix.png"}')

[SAVED] C:\RIE\outputs\DenseNet121\confusion_matrix.png


---
## 12 — Save Outputs

In [24]:
metrics_dict = {
    'model': 'DenseNet121', 'mode': RUN_MODE.upper(), 'best_epoch': best_epoch,
    'accuracy': round(acc, 4), 'precision': round(prec, 4),
    'recall': round(rec, 4), 'f1_score': round(f1, 4), 'roc_auc': round(auc, 4),
}
pd.DataFrame([metrics_dict]).to_csv(OUTPUT_DIR / 'validation_metrics.csv', index=False)

history_df = pd.DataFrame(history)
history_df.index = history_df.index + 1
history_df.index.name = 'epoch'
history_df.to_csv(OUTPUT_DIR / 'training_history.csv')

print(f'[SAVED] validation_metrics.csv, training_history.csv')
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name:35s} ({f.stat().st_size/1024:>8.1f} KB)')

[SAVED] validation_metrics.csv, training_history.csv
  best_model.pth                      ( 27804.6 KB)
  confusion_matrix.png                (   105.6 KB)
  training_curves.png                 (    91.8 KB)
  training_history.csv                (     4.0 KB)
  validation_metrics.csv              (     0.1 KB)


---
## 13 — TTA Evaluation

In [25]:
# TTA Evaluation — 5 augmented views per image
tta_transforms_list = [
    val_transforms,
    transforms.Compose([CLAHETransform(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]),
    transforms.Compose([CLAHETransform(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(degrees=(7, 7)), transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]),
    transforms.Compose([CLAHETransform(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(degrees=(-7, -7)), transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]),
    transforms.Compose([CLAHETransform(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(brightness=0.15), transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]),
]

def run_tta_inference(model, df_val, tta_list, device, batch_size=16):
    model.eval()
    all_variant_probs = []
    tta_labels = None
    for v_idx, t in enumerate(tta_list):
        ds = ChestXrayDataset(df_val, transform=t)
        dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)
        v_probs, v_labels = [], []
        with torch.no_grad():
            for images, labels, _ in dl:
                images = images.to(device, non_blocking=True)
                probs = torch.softmax(model(images), dim=1).cpu().numpy()
                v_probs.extend(probs)
                if v_idx == 0:
                    v_labels.extend(labels.numpy().flatten().tolist())
        all_variant_probs.append(np.array(v_probs, dtype=np.float32))
        if v_idx == 0:
            tta_labels = np.array(v_labels, dtype=np.int64)
        print(f'  Variant {v_idx} done | mean_prob={np.mean(v_probs):.4f}')
    avg_probs = np.stack(all_variant_probs, axis=0).mean(axis=0)
    return avg_probs, tta_labels

checkpoint = torch.load(best_model_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Running TTA with {len(tta_transforms_list)} variants...')

tta_probs, tta_labels = run_tta_inference(model, df_val, tta_transforms_list, DEVICE, BATCH_SIZE)

tta_roc = roc_auc_score(tta_labels, tta_probs, multi_class='ovr', average='macro')
tta_pr = average_precision_score(tta_labels, tta_probs, average='macro')


tta_preds = tta_probs.argmax(axis=1)
tta_prec = precision_score(tta_labels, tta_preds, average='macro', zero_division=0)
tta_rec = recall_score(tta_labels, tta_preds, average='macro', zero_division=0)
best_t_tta, best_f1_tta = 0.5, 0.0
print('-' * 60)
print(f'TTA Best Threshold  : {best_t_tta:.3f}')
print(f'TTA Precision       : {tta_prec:.4f}')
print(f'TTA Recall          : {tta_rec:.4f}')
cm_tta = confusion_matrix(tta_labels, tta_preds)
diag_tta = cm_tta.diagonal().sum()
total_tta = cm_tta.sum()
tta_acc = diag_tta / total_tta
print(f'TTA Accuracy: {tta_acc:.4f}')
print('=' * 70)

tta_metrics = {
    'model': 'DenseNet121', 'evaluation': 'TTA_5variants',
    'best_epoch': best_epoch,
    'tta_roc_auc': round(float(tta_roc), 4), 'tta_pr_auc': round(float(tta_pr), 4),
    'tta_f1': round(float(best_f1_tta), 4), 'tta_threshold': round(float(best_t_tta), 3),
    'tta_precision': round(float(tta_prec), 4), 'tta_recall': round(float(tta_rec), 4),
    'tta_accuracy': round(float(tta_acc), 4),
}
with open(OUTPUT_DIR / 'tta_metrics.json', 'w') as f:
    json.dump(tta_metrics, f, indent=2)
print(f'[SAVED] {OUTPUT_DIR / "tta_metrics.json"}')

Running TTA with 5 variants...
  Variant 0 done | mean_prob=0.0833
  Variant 1 done | mean_prob=0.0833
  Variant 2 done | mean_prob=0.0833
  Variant 3 done | mean_prob=0.0833
  Variant 4 done | mean_prob=0.0833
------------------------------------------------------------
TTA Best Threshold  : 0.500
TTA Precision       : 0.3562
TTA Recall          : 0.3484
TTA Accuracy: 0.3501
[SAVED] C:\RIE\outputs\DenseNet121\tta_metrics.json
